# Cell 0 - Final Falconsai T5 Text-to-Bullets Experiment

This notebook trains **`Falconsai/text_summarization`** on the final 10,000-row dataset and evaluates it correctly.

The main design choices are:

- **Architecture:** T5 encoder-decoder / Seq2Seq.
- **Input capacity:** `2048` tokens.
- **Train batch size:** `16`.
- **Eval batch size:** `16`.
- **Epochs:** `7`.
- **Split:** stratified by `source` and bullet-count bucket so train/validation preserve the important output distribution.
- **Training-time generation validation:** one **fixed, strong 100-example subset** selected from the validation pool. It deliberately covers bullet counts from **1 through 10** and all source categories. Using the same 100 examples each epoch makes checkpoint comparisons meaningful.
- **Bullet boundary handling:** T5's SentencePiece tokenizer does not reliably preserve newlines. The notebook therefore teaches a dedicated `<BULLET>` token internally and converts it back to real `- ` lines at inference/evaluation time. This fixes the earlier one-bullet metric problem.
- **Per-epoch metrics:** validation loss, ROUGE-1/2/L, BERTScore P/R/F1, bullet-format score, predicted/reference bullet counts, bullet-count error, compression ratios, and generated-token count.
- **Best checkpoint:** highest validation ROUGE-L on the fixed 100-example challenge set.
- **Final evaluation:** a completely separate CSV is used only after training.
- **Final comparison:** the T5 specialist is benchmarked on exactly the same final evaluation rows as the fine-tuned SmolLM2-135M, original SmolLM2-135M-Instruct, and Qwen3-0.6B.

In [ ]:
# Cell 1 - Install dependencies
# Do NOT upgrade numpy/pandas/scikit-learn supplied by the notebook runtime.

!pip install -q -U \
    transformers \
    datasets \
    accelerate \
    sentencepiece \
    tqdm \
    rouge-score \
    bert-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.1 MB/s eta 0:00:00


In [ ]:
# Cell 2 - Imports

import os
import gc
import time
import glob
import random
import warnings

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

from rouge_score import rouge_scorer
from bert_score import BERTScorer

warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
# Cell 3 - Reproducibility and main configuration

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = 'Falconsai/text_summarization'

# The notebook checks both names so it works with the current uploaded final file
# or a cleaner renamed copy in Colab.
TRAIN_CSV_CANDIDATES = [
    '/content/training_bullet_dataset_10000 (2).csv',
]

TRAIN_CSV = next(
    (path for path in TRAIN_CSV_CANDIDATES if os.path.exists(path)),
    TRAIN_CSV_CANDIDATES[-1],
)

# This must be a SEPARATE untouched evaluation CSV with the same columns:
# text, source, example_id, bullet_points
EVAL_CSV = '/content/output_with_bullet_points.csv'

OUTPUT_DIR = '/content/falconsai-t5-bullet-training'
FINAL_MODEL_DIR = '/content/falconsai-t5-bullet-specialist'

MAX_INPUT_LENGTH = 2048
MAX_TARGET_LENGTH = 256
NUM_EPOCHS = 7

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1

VALIDATION_FRACTION = 0.10
EPOCH_EVAL_SIZE = 100

# Internal delimiter used because T5/SentencePiece can normalize newlines.
BULLET_TOKEN = '<BULLET>'

print('Seed:', SEED)
print('Training CSV:', TRAIN_CSV)
print('Final eval CSV:', EVAL_CSV)

Seed: 42
Training CSV: /content/training_bullet_dataset_10000 (2).csv
Final eval CSV: /content/output_with_bullet_points.csv


In [ ]:
# Cell 4 - GPU check

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime in Google Colab before training.')

print('GPU:', torch.cuda.get_device_name(0))
print(
    'VRAM:',
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    'GB',
)

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
# Cell 5 - Load the final 10,000-row training dataset

train_full_df = pd.read_csv(TRAIN_CSV)

print('Rows:', len(train_full_df))
print('Columns:', train_full_df.columns.tolist())
display(train_full_df.head())

Rows: 10000
Columns: ['text', 'source', 'example_id', 'bullet_points']


,text,source,example_id,bullet_points
0,The writer mentioned that the Riverton room wa...,news,tb_00001,"- Public comments are due by February 9, 2026."
1,The study tracked 917 participants for 9 month...,research_note,tb_00002,- The study tracked 917 participants for 9 mon...
2,The final sentence asked readers not to reply-...,review,tb_00003,- Battery life fell short of the advertised 88...
3,"Hi team, I am sending one consolidated note be...",email,tb_00004,- Only managers may approve changes after Apri...
4,Support thread summary for a customer contacti...,customer_support,tb_00005,- The agent escalated the ticket after two fai...


In [ ]:
# Cell 6 - Validate schema and minimally clean data

REQUIRED_COLUMNS = ['text', 'source', 'example_id', 'bullet_points']

for column in REQUIRED_COLUMNS:
    assert column in train_full_df.columns, f'Missing required column: {column}'


def clean_dataframe(df):
    df = df.copy()
    df = df.dropna(subset=['text', 'bullet_points'])
    df['text'] = df['text'].astype(str).str.strip()
    df['source'] = df['source'].astype(str).str.strip()
    df['example_id'] = df['example_id'].astype(str).str.strip()
    df['bullet_points'] = df['bullet_points'].astype(str).str.strip()
    df = df[(df['text'] != '') & (df['bullet_points'] != '')]
    return df.reset_index(drop=True)


train_full_df = clean_dataframe(train_full_df)

assert len(train_full_df) == 10000, (
    f'Expected 10,000 valid rows, found {len(train_full_df)}.'
)

print('Clean rows:', len(train_full_df))
print('Exact duplicate source texts:', train_full_df['text'].duplicated().sum())
print('Exact duplicate IDs:', train_full_df['example_id'].duplicated().sum())

Clean rows: 10000
Exact duplicate source texts: 0
Exact duplicate IDs: 0


In [ ]:
# Cell 9 - Task instruction
#
# T5 is not a chat model. This instruction is prepended to the source text
# and sent to the encoder.

TASK_INSTRUCTION = """Convert the following English text into concise bullet points containing all materially important information.

Follow these rules:

- Extract all important and independently useful points.
- The number of bullets must depend entirely on the information in the text.
- Never use a fixed number of bullets.
- Use one bullet for each distinct important point.
- Combine details that naturally belong together.
- Remove repetition, filler, metadata, boilerplate, and trivial details.
- Do not repeat the same information in multiple bullets.
- Preserve important names, dates, numbers, quantities, comparisons, causes, conditions, decisions, and conclusions.
- Do not add, infer, or assume information that is not supported by the source text.
- Do not turn contextual information into new advice or recommendations.
- Keep every bullet concise while preserving the original meaning.
- Return only bullet points.""".strip()

print(TASK_INSTRUCTION)

Convert the following English text into concise bullet points containing all materially important information.

Follow these rules:

- Extract all important and independently useful points.
- The number of bullets must depend entirely on the information in the text.
- Never use a fixed number of bullets.
- Use one bullet for each distinct important point.
- Combine details that naturally belong together.
- Remove repetition, filler, metadata, boilerplate, and trivial details.
- Do not repeat the same information in multiple bullets.
- Preserve important names, dates, numbers, quantities, comparisons, causes, conditions, decisions, and conclusions.
- Do not add, infer, or assume information that is not supported by the source text.
- Do not turn contextual information into new advice or recommendations.
- Keep every bullet concise while preserving the original meaning.
- Return only bullet points.


In [ ]:
# Cell 7 - Bullet-count and source-distribution analysis


def count_bullets(text):
    return sum(
        line.strip().startswith('- ')
        for line in str(text).splitlines()
        if line.strip()
    )


train_full_df['bullet_count'] = (
    train_full_df['bullet_points'].apply(count_bullets)
)
train_full_df['source_words'] = (
    train_full_df['text'].str.split().str.len()
)
train_full_df['target_words'] = (
    train_full_df['bullet_points'].str.split().str.len()
)
train_full_df['reference_compression'] = (
    train_full_df['target_words'] / train_full_df['source_words'].clip(lower=1)
)

print('BULLET COUNT DISTRIBUTION')
print(train_full_df['bullet_count'].value_counts().sort_index())
print('\nAverage bullets:', round(train_full_df['bullet_count'].mean(), 3))

print('\nSOURCE DISTRIBUTION')
print(train_full_df['source'].value_counts().sort_index())

print('\nSOURCE WORD LENGTHS')
print(
    train_full_df['source_words'].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print('\nREFERENCE COMPRESSION')
print(
    train_full_df['reference_compression'].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

BULLET COUNT DISTRIBUTION
bullet_count
1     1248
2     1800
3     2271
4     1949
5     1259
6      764
7      423
8      201
9       76
10       9
Name: count, dtype: int64

Average bullets: 3.568

SOURCE DISTRIBUTION
source
announcement               500
business_update            500
customer_support           500
educational_explanation    500
email                      500
financial_report           500
forum_discussion           500
general_info               500
incident_report            500
instructions               500
meeting_notes              500
news                       501
policy                     500
product_info               500
project_update             500
qa                         500
research_note              500
review                     500
technical_explanation      499
workplace_communication    500
Name: count, dtype: int64

SOURCE WORD LENGTHS
count    10000.000000
mean       253.887200
std        238.973455
min         27.000000
10%         63.000

In [ ]:
# Cell 8 - Load tokenizer and add a dedicated bullet-boundary token
#
# WHY THIS MATTERS
# ----------------
# T5's SentencePiece tokenizer can normalize line breaks into spaces.
# If we train directly on:
#
#   - point one\n- point two\n- point three
#
# the decoded model output can become:
#
#   - point one - point two - point three
#
# which looks like one line and breaks bullet-count evaluation.
#
# We therefore train internally as:
#
#   <BULLET> point one <BULLET> point two <BULLET> point three
#
# and convert <BULLET> back to real newline bullets after generation.

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if BULLET_TOKEN not in tokenizer.get_vocab():
    num_added = tokenizer.add_tokens([BULLET_TOKEN])
else:
    num_added = 0

print('Tokenizer vocabulary size:', len(tokenizer))
print('Added bullet tokens:', num_added)
print('Bullet token ID:', tokenizer.convert_tokens_to_ids(BULLET_TOKEN))

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Tokenizer vocabulary size: 32101
Added bullet tokens: 1
Bullet token ID: 32100


In [ ]:
# Cell 10 - Serialize and postprocess bullet targets


def parse_bullet_text(text):
    """Return clean bullet contents without the '- ' prefix."""
    bullets = []
    for line in str(text).splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith('- '):
            bullets.append(line[2:].strip())
        else:
            # Dataset QC should make this rare; retaining the line is safer than dropping it.
            bullets.append(line)
    return [b for b in bullets if b]


def serialize_target(text):
    """Convert normal multiline bullets into T5-friendly bullet-token format."""
    bullets = parse_bullet_text(text)
    return ' '.join(f'{BULLET_TOKEN} {bullet}' for bullet in bullets)


def postprocess_generated_text(text):
    """Convert T5's internal <BULLET> representation back to real bullet lines."""
    text = str(text).strip()

    if BULLET_TOKEN in text:
        pieces = [piece.strip() for piece in text.split(BULLET_TOKEN) if piece.strip()]
    else:
        # Fallback for generations that fail to emit the learned delimiter.
        pieces = parse_bullet_text(text)
        if not pieces and text:
            pieces = [text.lstrip('- ').strip()]

    cleaned = []
    for piece in pieces:
        piece = piece.strip()
        if piece.startswith('- '):
            piece = piece[2:].strip()
        if piece:
            cleaned.append(piece)

    return '\n'.join(f'- {piece}' for piece in cleaned)


sample_target = train_full_df.iloc[0]['bullet_points']
print('ORIGINAL TARGET:\n', sample_target)
print('\nSERIALIZED TARGET:\n', serialize_target(sample_target))
print('\nROUND TRIP:\n', postprocess_generated_text(serialize_target(sample_target)))

ORIGINAL TARGET:
 - Public comments are due by February 9, 2026.

SERIALIZED TARGET:
 <BULLET> Public comments are due by February 9, 2026.

ROUND TRIP:
 - Public comments are due by February 9, 2026.


In [ ]:
# Cell 11 - Verify the 2048-token encoder budget on the FINAL dataset

input_lengths = []

for text in tqdm(train_full_df['text'], desc='Measuring encoder lengths'):
    full_input = TASK_INSTRUCTION + '\n\nText:\n' + text
    ids = tokenizer(
        full_input,
        add_special_tokens=True,
        truncation=False,
    )['input_ids']
    input_lengths.append(len(ids))

input_length_series = pd.Series(input_lengths)

print(
    input_length_series.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

num_over = int((input_length_series > MAX_INPUT_LENGTH).sum())
print('\nExamples over 2048 tokens:', num_over)
print('Percentage:', round(num_over / len(input_length_series) * 100, 3), '%')

# Also verify decoder-target length AFTER <BULLET> serialization.
target_lengths = []
for target in tqdm(train_full_df['bullet_points'], desc='Measuring target lengths'):
    serialized = serialize_target(target)
    ids = tokenizer(
        serialized,
        add_special_tokens=True,
        truncation=False,
    )['input_ids']
    target_lengths.append(len(ids))

target_length_series = pd.Series(target_lengths)
print('\nTARGET TOKEN LENGTHS')
print(
    target_length_series.describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

targets_over = int((target_length_series > MAX_TARGET_LENGTH).sum())
print('Targets over 256 tokens:', targets_over)
print('Percentage:', round(targets_over / len(target_length_series) * 100, 3), '%')


Measuring encoder lengths:   0%|          | 0/10000 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (961 > 512). Running this sequence through the model will result in indexing errors


count    10000.000000
mean       514.335200
std        299.550219
min        229.000000
50%        401.000000
75%        596.000000
90%        972.000000
95%       1070.350000
99%       1497.000000
max       1566.000000
dtype: float64

Examples over 2048 tokens: 0
Percentage: 0.0 %


Measuring target lengths:   0%|          | 0/10000 [00:00<?, ?it/s]


TARGET TOKEN LENGTHS
count    10000.000000
mean        51.883400
std         26.755687
min          7.000000
50%         49.000000
75%         68.000000
90%         89.000000
95%        102.000000
99%        127.000000
max        175.000000
dtype: float64
Targets over 256 tokens: 0
Percentage: 0.0 %


In [ ]:
# Cell 12 - Stratified 90/10 train-validation split
#
# We stratify on:
#
#   source category + bullet-count bucket
#
# This preserves both domain and output-granularity distributions.
# Rare high bullet counts are pooled into '6+' for the split so every stratum
# has enough examples. Exact counts are still preserved in the data itself.


def split_bullet_bucket(n):
    return str(int(n)) if int(n) <= 5 else '6+'


train_full_df['split_bullet_bucket'] = (
    train_full_df['bullet_count'].apply(split_bullet_bucket)
)

stratify_key = (
    train_full_df['source'].astype(str)
    + '__'
    + train_full_df['split_bullet_bucket'].astype(str)
)

train_df, validation_pool_df = train_test_split(
    train_full_df,
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
    stratify=stratify_key,
)

train_df = train_df.reset_index(drop=True)
validation_pool_df = validation_pool_df.reset_index(drop=True)

print('Training rows:', len(train_df))
print('Validation pool rows:', len(validation_pool_df))

print('\nTRAIN BULLET DISTRIBUTION (%)')
print((train_df['bullet_count'].value_counts(normalize=True).sort_index() * 100).round(2))

print('\nVALIDATION BULLET DISTRIBUTION (%)')
print((validation_pool_df['bullet_count'].value_counts(normalize=True).sort_index() * 100).round(2))

print('\nTRAIN SOURCE DISTRIBUTION')
print(train_df['source'].value_counts().sort_index())

print('\nVALIDATION SOURCE DISTRIBUTION')
print(validation_pool_df['source'].value_counts().sort_index())

# Verify source-length distribution is also preserved approximately.
print('\nTRAIN SOURCE-LENGTH QUANTILES')
print(train_df['source_words'].quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))
print('\nVALIDATION SOURCE-LENGTH QUANTILES')
print(validation_pool_df['source_words'].quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))


Training rows: 9000
Validation pool rows: 1000

TRAIN BULLET DISTRIBUTION (%)
bullet_count
1     12.47
2     18.01
3     22.72
4     19.50
5     12.59
6      7.66
7      4.13
8      2.03
9      0.80
10     0.09
Name: proportion, dtype: float64

VALIDATION BULLET DISTRIBUTION (%)
bullet_count
1     12.6
2     17.9
3     22.6
4     19.4
5     12.6
6      7.5
7      5.1
8      1.8
9      0.4
10     0.1
Name: proportion, dtype: float64

TRAIN SOURCE DISTRIBUTION
source
announcement               451
business_update            450
customer_support           450
educational_explanation    450
email                      450
financial_report           450
forum_discussion           451
general_info               450
incident_report            451
instructions               450
meeting_notes              450
news                       452
policy                     451
product_info               449
project_update             451
qa                         449
research_note              449
rev

In [ ]:
# Cell 13 - Build a FIXED strong 100-example epoch-evaluation set
#
# IMPORTANT:
# We intentionally do NOT choose a fresh random 100 every epoch.
# Different examples every epoch make checkpoint scores noisy and incomparable.
#
# Instead we build one fixed challenge set from the held-out validation pool that:
#   - contains all 20 source categories,
#   - deliberately covers bullet counts 1 through 10,
#   - over-represents rarer 6-10 bullet examples,
#   - remains completely outside gradient updates.
#
# Requested bullet-size quotas sum to exactly 100.

EVAL_BULLET_QUOTAS = {
    '1': 12,
    '2': 12,
    '3': 14,
    '4': 14,
    '5': 12,
    '6': 12,
    '7': 10,
    '8': 9,
    '9+': 5,
}


def eval_bullet_bucket(n):
    n = int(n)
    return str(n) if n <= 8 else '9+'


def build_balanced_epoch_eval(validation_df, seed=42):
    rng = np.random.default_rng(seed)
    work = validation_df.copy()
    work['eval_bullet_bucket'] = work['bullet_count'].apply(eval_bullet_bucket)

    selected_indices = []

    for bucket, quota in EVAL_BULLET_QUOTAS.items():
        candidates = work[work['eval_bullet_bucket'] == bucket].copy()

        if len(candidates) < quota:
            raise ValueError(
                f'Not enough validation examples for bullet bucket {bucket}: '
                f'need {quota}, found {len(candidates)}.'
            )

        # Randomize within each source, then choose round-robin across sources.
        # This prevents one domain from dominating a bullet-count bucket.
        candidates['_rand'] = rng.random(len(candidates))
        groups = {
            source: group.sort_values('_rand').index.tolist()
            for source, group in candidates.groupby('source')
        }

        sources = list(groups.keys())
        rng.shuffle(sources)

        chosen_for_bucket = []
        while len(chosen_for_bucket) < quota:
            made_progress = False
            for source in sources:
                if groups[source] and len(chosen_for_bucket) < quota:
                    chosen_for_bucket.append(groups[source].pop(0))
                    made_progress = True
            if not made_progress:
                break

        if len(chosen_for_bucket) != quota:
            raise RuntimeError(
                f'Could only choose {len(chosen_for_bucket)} examples for bucket {bucket}.'
            )

        selected_indices.extend(chosen_for_bucket)

    result = work.loc[selected_indices].copy().reset_index(drop=True)
    return result


EPOCH_EVAL_RAW = build_balanced_epoch_eval(validation_pool_df, seed=SEED)

assert len(EPOCH_EVAL_RAW) == EPOCH_EVAL_SIZE
assert EPOCH_EVAL_RAW['source'].nunique() == 20

print('Epoch-evaluation examples:', len(EPOCH_EVAL_RAW))
print('Source categories represented:', EPOCH_EVAL_RAW['source'].nunique())

print('\nEXACT BULLET COUNTS')
print(EPOCH_EVAL_RAW['bullet_count'].value_counts().sort_index())

print('\nSOURCE COUNTS')
print(EPOCH_EVAL_RAW['source'].value_counts().sort_index())

In [ ]:
# Cell 14 - Save the split and the fixed 100-example challenge set
#
# Saving these is important for reproducibility. Every rerun can use the exact same
# train/validation/challenge rows rather than silently changing checkpoint selection.

train_df.to_csv('/content/final_train_split.csv', index=False)
validation_pool_df.to_csv('/content/final_validation_pool.csv', index=False)
EPOCH_EVAL_RAW.to_csv('/content/fixed_epoch_eval_100.csv', index=False)

print('Saved:')
print('/content/final_train_split.csv')
print('/content/final_validation_pool.csv')
print('/content/fixed_epoch_eval_100.csv')

Saved:
/content/final_train_split.csv
/content/final_validation_pool.csv
/content/fixed_epoch_eval_100.csv


In [ ]:
# Cell 15 - Convert pandas frames to Hugging Face Dataset objects

train_dataset_raw = Dataset.from_pandas(
    train_df,
    preserve_index=False,
)

epoch_eval_dataset_raw = Dataset.from_pandas(
    EPOCH_EVAL_RAW,
    preserve_index=False,
)

print(train_dataset_raw)
print(epoch_eval_dataset_raw)

Dataset({
    features: ['text', 'source', 'example_id', 'bullet_points', 'bullet_count', 'source_words', 'target_words', 'reference_compression', 'split_bullet_bucket'],
    num_rows: 9000
})
Dataset({
    features: ['text', 'source', 'example_id', 'bullet_points', 'bullet_count', 'source_words', 'target_words', 'reference_compression', 'split_bullet_bucket', 'eval_bullet_bucket'],
    num_rows: 100
})


In [ ]:
# Cell 16 - Seq2Seq preprocessing
#
# ENCODER INPUT:
#   task instruction + source document
#
# DECODER TARGET:
#   <BULLET> fact 1 <BULLET> fact 2 ...
#
# The dedicated bullet token gives T5 an unambiguous structure to learn.


def preprocess_batch(examples):
    inputs = [
        TASK_INSTRUCTION + '\n\nText:\n' + text
        for text in examples['text']
    ]

    serialized_targets = [
        serialize_target(target)
        for target in examples['bullet_points']
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )

    labels = tokenizer(
        text_target=serialized_targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

In [ ]:
# Cell 17 - Tokenize train and fixed epoch-evaluation datasets
#
# The resulting datasets MUST contain only model-facing fields.

train_dataset = train_dataset_raw.map(
    preprocess_batch,
    batched=True,
    remove_columns=train_dataset_raw.column_names,
    desc='Tokenizing training data',
)

epoch_eval_dataset = epoch_eval_dataset_raw.map(
    preprocess_batch,
    batched=True,
    remove_columns=epoch_eval_dataset_raw.column_names,
    desc='Tokenizing fixed epoch evaluation data',
)

print('Train columns:', train_dataset.column_names)
print('Eval columns:', epoch_eval_dataset.column_names)

required_model_columns = {'input_ids', 'attention_mask', 'labels'}
assert required_model_columns.issubset(train_dataset.column_names)
assert required_model_columns.issubset(epoch_eval_dataset.column_names)

Tokenizing training data:   0%|          | 0/9000 [00:00<?, ? examples/s]

Tokenizing fixed epoch evaluation data:   0%|          | 0/100 [00:00<?, ? examples/s]

Train columns: ['input_ids', 'attention_mask', 'labels']
Eval columns: ['input_ids', 'attention_mask', 'labels']


In [ ]:
# Cell 18 - Load Falconsai T5 and resize embeddings for <BULLET>

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

# We added one vocabulary item to the tokenizer, so the model needs a matching
# input/output embedding row.
model.resize_token_embeddings(len(tokenizer))

# Gradient checkpointing reduces activation memory and makes physical batch 16
# safer on a T4 for the long-tail examples.
model.gradient_checkpointing_enable()
model.config.use_cache = False

print('Encoder-decoder:', model.config.is_encoder_decoder)

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('Parameters:', f'{num_params:,}')
print('Trainable:', f'{trainable_params:,}')
print('Trainable %:', round(trainable_params / num_params * 100, 2))

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Encoder-decoder: True
Parameters: 60,492,800
Trainable: 60,492,800
Trainable %: 100.0


In [ ]:
# Cell 19 - Dynamic Seq2Seq padding
#
# MAX_INPUT_LENGTH=2048 is only an upper bound. Dynamic padding means each batch
# is padded to the longest sequence actually present in that batch, not 2048.

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

In [ ]:
# Cell 20 - Metric helpers

rouge = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True,
)


def bullet_format_score(text):
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]
    if not lines:
        return 0.0
    return sum(line.startswith('- ') for line in lines) / len(lines)


def word_count(text):
    return len(str(text).split())


def sanitize_prediction_ids(predictions):
    """Prevent tokenizer OverflowError from -100/out-of-range generated IDs."""
    predictions = np.asarray(predictions)
    valid = (predictions >= 0) & (predictions < len(tokenizer))
    return np.where(valid, predictions, tokenizer.pad_token_id).astype(np.int64)

In [ ]:
# Cell 21 - BERTScore evaluator
#
# We use a lighter persistent encoder during per-epoch evaluation so batch-16 T5
# training keeps enough GPU headroom. The SAME scorer is reused for every model in
# final comparison, so BERTScore remains apples-to-apples.

bert_scorer = BERTScorer(
    model_type='distilbert-base-uncased',
    lang='en',
    device='cuda',
    rescale_with_baseline=False,
)

print('BERTScore evaluator ready.')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore evaluator ready.


In [ ]:
# Cell 22 - Complete generation metrics for the fixed 100-example challenge set
#
# IMPORTANT FIXES compared with the earlier notebook:
# 1. The references come directly from EPOCH_EVAL_RAW, not by decoding T5 labels.
#    This preserves true newline bullet boundaries.
# 2. Predictions use the learned <BULLET> token and are postprocessed into real lines.
# 3. Generated token IDs are sanitized before decoding to avoid OverflowError.


def compute_metrics(eval_prediction):
    predictions, _labels = eval_prediction

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions_for_decode = sanitize_prediction_ids(predictions)

    raw_decoded = tokenizer.batch_decode(
        predictions_for_decode,
        skip_special_tokens=True,
    )

    decoded_predictions = [
        postprocess_generated_text(text)
        for text in raw_decoded
    ]

    decoded_references = EPOCH_EVAL_RAW['bullet_points'].tolist()
    source_texts = EPOCH_EVAL_RAW['text'].tolist()

    if len(decoded_predictions) != len(decoded_references):
        raise RuntimeError(
            f'Prediction/reference length mismatch: '
            f'{len(decoded_predictions)} vs {len(decoded_references)}'
        )

    rouge1_values = []
    rouge2_values = []
    rougeL_values = []

    for reference, prediction in zip(decoded_references, decoded_predictions):
        scores = rouge.score(reference, prediction)
        rouge1_values.append(scores['rouge1'].fmeasure)
        rouge2_values.append(scores['rouge2'].fmeasure)
        rougeL_values.append(scores['rougeL'].fmeasure)

    P, R, F1 = bert_scorer.score(
        decoded_predictions,
        decoded_references,
    )

    format_values = [bullet_format_score(x) for x in decoded_predictions]
    predicted_bullets = [count_bullets(x) for x in decoded_predictions]
    reference_bullets = [count_bullets(x) for x in decoded_references]
    bullet_count_errors = [
        abs(pred - ref)
        for pred, ref in zip(predicted_bullets, reference_bullets)
    ]

    compression_values = []
    reference_compression_values = []

    for source, prediction, reference in zip(
        source_texts,
        decoded_predictions,
        decoded_references,
    ):
        source_words = max(word_count(source), 1)
        compression_values.append(word_count(prediction) / source_words)
        reference_compression_values.append(word_count(reference) / source_words)

    generated_lengths = [
        int(np.sum(sequence != tokenizer.pad_token_id))
        for sequence in predictions_for_decode
    ]

    return {
        'rouge1': float(np.mean(rouge1_values)),
        'rouge2': float(np.mean(rouge2_values)),
        'rougeL': float(np.mean(rougeL_values)),
        'bertscore_precision': float(P.mean().item()),
        'bertscore_recall': float(R.mean().item()),
        'bertscore_f1': float(F1.mean().item()),
        'bullet_format': float(np.mean(format_values)),
        'avg_predicted_bullets': float(np.mean(predicted_bullets)),
        'avg_reference_bullets': float(np.mean(reference_bullets)),
        'mean_bullet_count_error': float(np.mean(bullet_count_errors)),
        'compression_ratio': float(np.mean(compression_values)),
        'reference_compression_ratio': float(np.mean(reference_compression_values)),
        'avg_generated_tokens': float(np.mean(generated_lengths)),
    }

In [ ]:
# Cell 23 - Training configuration
#
# Requested physical batches:
#   train = 16
#   eval  = 16
#
# Evaluation generates only the fixed 100-example challenge set at the end of
# each epoch. Every epoch checkpoint is preserved.

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=50,
    lr_scheduler_type='cosine',

    fp16=True,
    bf16=False,

    eval_strategy='epoch',
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=1,

    save_strategy='epoch',
    save_total_limit=None,

    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    greater_is_better=True,

    logging_steps=25,
    eval_accumulation_steps=1,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    seed=SEED,
    data_seed=SEED,
    report_to='none',
)

In [ ]:
# Cell 24 - Create Trainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=epoch_eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Trainer ready.')
print('Train columns:', trainer.train_dataset.column_names)
print('Eval columns:', trainer.eval_dataset.column_names)

Trainer ready.
Train columns: ['input_ids', 'attention_mask', 'labels']
Eval columns: ['input_ids', 'attention_mask', 'labels']


In [ ]:
# Cell 25 - Batch-16 forward/backward VRAM smoke test
#
# This catches an OOM before a multi-hour run. It uses a real training backward pass.
# If it fails, the notebook deliberately raises an informative error rather than
# silently changing the requested batch size.

test_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
)

outputs = None
batch = next(iter(test_loader))
batch = {key: value.to('cuda') for key, value in batch.items()}
model = model.to('cuda')
model.train()

try:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    outputs = model(**batch)
    outputs.loss.backward()

    peak_gb = torch.cuda.max_memory_allocated() / 1024**3
    print('Batch-16 test loss:', float(outputs.loss.detach().cpu()))
    print('Peak allocated GPU memory:', round(peak_gb, 2), 'GB')
    print('Batch size 16 smoke test PASSED.')

    model.zero_grad(set_to_none=True)

except torch.cuda.OutOfMemoryError as exc:
    model.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    raise RuntimeError(
        'Physical train batch 16 does not fit this runtime. '
        'If this happens on a T4, change TRAIN_BATCH_SIZE to 8 and '
        'GRADIENT_ACCUMULATION_STEPS to 2 to preserve effective batch 16.'
    ) from exc

finally:
    del batch
    if outputs is not None:
        del outputs
    del test_loader
    gc.collect()
    torch.cuda.empty_cache()
    model.eval()

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Batch-16 test loss: 3.8240861892700195
Peak allocated GPU memory: 3.09 GB
Batch size 16 smoke test PASSED.


In [ ]:
# Cell 26 - Train for 7 epochs
#
# At the end of every epoch the SAME balanced 100 examples are generated and scored.
# This lets us compare checkpoints without sample noise.

start_time = time.time()
train_result = trainer.train()
training_seconds = time.time() - start_time

print()
print('Training time:', round(training_seconds / 60, 2), 'minutes')

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Bertscore Precision,Bertscore Recall,Bertscore F1,Bullet Format,Avg Predicted Bullets,Avg Reference Bullets,Mean Bullet Count Error,Compression Ratio,Reference Compression Ratio,Avg Generated Tokens
1,0.512144,0.422162,0.800679,0.784985,0.777048,0.962636,0.931595,0.945735,1.000000,1.240000,4.540000,3.360000,0.221405,0.250104,54.470000
2,0.323194,0.289997,0.954405,0.950253,0.949321,0.996247,0.984182,0.990020,1.000000,4.130000,4.540000,0.450000,0.241440,0.250104,60.860000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Bertscore Precision,Bertscore Recall,Bertscore F1,Bullet Format,Avg Predicted Bullets,Avg Reference Bullets,Mean Bullet Count Error,Compression Ratio,Reference Compression Ratio,Avg Generated Tokens
1,0.512144,0.422162,0.800679,0.784985,0.777048,0.962636,0.931595,0.945735,1.000000,1.240000,4.540000,3.360000,0.221405,0.250104,54.470000
2,0.323194,0.289997,0.954405,0.950253,0.949321,0.996247,0.984182,0.990020,1.000000,4.130000,4.540000,0.450000,0.241440,0.250104,60.860000
3,0.262950,0.217332,0.975055,0.972488,0.970062,0.997629,0.991013,0.994240,1.000000,4.370000,4.540000,0.290000,0.246384,0.250104,64.470000
4,0.217195,0.178819,0.981595,0.979642,0.977414,0.998238,0.992775,0.995441,1.000000,4.380000,4.540000,0.220000,0.246970,0.250104,64.360000
5,0.193471,0.161905,0.986262,0.984476,0.982839,0.998394,0.994260,0.996282,1.000000,4.420000,4.540000,0.180000,0.248092,0.250104,64.990000
6,0.190032,0.156321,0.986262,0.984476,0.982839,0.998394,0.994260,0.996282,1.000000,4.420000,4.540000,0.180000,0.248092,0.250104,64.990000
7,0.187106,0.155510,0.985714,0.983724,0.982285,0.998359,0.994057,0.996163,1.000000,4.420000,4.540000,0.180000,0.247879,0.250104,64.830000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



Training time: 135.92 minutes


In [ ]:
# Cell 27 - Full training history and epoch-by-epoch generation leaderboard

history_df = pd.DataFrame(trainer.state.log_history)
display(history_df)

epoch_metrics = history_df[
    history_df.get('eval_rougeL', pd.Series(index=history_df.index, dtype=float)).notna()
].copy()

columns_to_show = [
    'epoch',
    'eval_loss',
    'eval_rouge1',
    'eval_rouge2',
    'eval_rougeL',
    'eval_bertscore_precision',
    'eval_bertscore_recall',
    'eval_bertscore_f1',
    'eval_bullet_format',
    'eval_avg_predicted_bullets',
    'eval_avg_reference_bullets',
    'eval_mean_bullet_count_error',
    'eval_compression_ratio',
    'eval_reference_compression_ratio',
    'eval_avg_generated_tokens',
]

existing_columns = [c for c in columns_to_show if c in epoch_metrics.columns]
epoch_leaderboard = epoch_metrics[existing_columns].sort_values(
    'eval_rougeL',
    ascending=False,
)

display(epoch_leaderboard)

epoch_leaderboard.to_csv('/content/t5_epoch_metrics.csv', index=False)

,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_rouge1,eval_rouge2,eval_rougeL,eval_bertscore_precision,...,eval_reference_compression_ratio,eval_avg_generated_tokens,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,3.792497,7.371298,9.600000e-06,0.044405,25,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3.213473,3.800910,1.960000e-05,0.088810,50,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2.604301,3.303050,1.999812e-05,0.133215,75,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2.202948,2.527774,1.999218e-05,0.177620,100,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.947727,2.442043,1.998216e-05,0.222025,125,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,0.191187,0.999095,1.462820e-08,6.882771,3875,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
161,0.188801,0.428116,5.749158e-09,6.927176,3900,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
162,0.187106,0.733309,9.419727e-10,6.971581,3925,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
163,NaN,NaN,NaN,7.000000,3941,0.15551,0.985714,0.983724,0.982285,0.998359,...,0.250104,64.83,18.295,5.466,0.383,NaN,NaN,NaN,NaN,NaN


,epoch,eval_loss,eval_rouge1,eval_rouge2,eval_rougeL,eval_bertscore_precision,eval_bertscore_recall,eval_bertscore_f1,eval_bullet_format,eval_avg_predicted_bullets,eval_avg_reference_bullets,eval_mean_bullet_count_error,eval_compression_ratio,eval_reference_compression_ratio,eval_avg_generated_tokens
116,5.0,0.161905,0.986262,0.984476,0.982839,0.998394,0.994260,0.996282,1.0,4.42,4.54,0.18,0.248092,0.250104,64.99
140,6.0,0.156321,0.986262,0.984476,0.982839,0.998394,0.994260,0.996282,1.0,4.42,4.54,0.18,0.248092,0.250104,64.99
163,7.0,0.155510,0.985714,0.983724,0.982285,0.998359,0.994057,0.996163,1.0,4.42,4.54,0.18,0.247879,0.250104,64.83
93,4.0,0.178819,0.981595,0.979642,0.977414,0.998238,0.992775,0.995441,1.0,4.38,4.54,0.22,0.246970,0.250104,64.36
69,3.0,0.217332,0.975055,0.972488,0.970062,0.997629,0.991013,0.994240,1.0,4.37,4.54,0.29,0.246384,0.250104,64.47
46,2.0,0.289997,0.954405,0.950253,0.949321,0.996247,0.984182,0.990020,1.0,4.13,4.54,0.45,0.241440,0.250104,60.86
22,1.0,0.422162,0.800679,0.784985,0.777048,0.962636,0.931595,0.945735,1.0,1.24,4.54,3.36,0.221405,0.250104,54.47


In [ ]:
# Cell 28 - Best checkpoint and all saved epoch checkpoints

print('Best checkpoint:', trainer.state.best_model_checkpoint)
print('Best validation ROUGE-L:', trainer.state.best_metric)

checkpoints = sorted(
    glob.glob(OUTPUT_DIR + '/checkpoint-*'),
    key=lambda path: int(path.split('-')[-1]),
)

print('\nSaved checkpoints:')
for i, checkpoint in enumerate(checkpoints, start=1):
    print(f'{i}: {checkpoint}')

Best checkpoint: /content/falconsai-t5-bullet-training/checkpoint-2815
Best validation ROUGE-L: 0.9828387067366637

Saved checkpoints:
1: /content/falconsai-t5-bullet-training/checkpoint-563
2: /content/falconsai-t5-bullet-training/checkpoint-1126
3: /content/falconsai-t5-bullet-training/checkpoint-1689
4: /content/falconsai-t5-bullet-training/checkpoint-2252
5: /content/falconsai-t5-bullet-training/checkpoint-2815
6: /content/falconsai-t5-bullet-training/checkpoint-3378
7: /content/falconsai-t5-bullet-training/checkpoint-3941


In [ ]:
# Cell 29 - Save the automatically restored best specialist
#
# load_best_model_at_end=True means trainer.model already contains the checkpoint
# with the highest validation ROUGE-L on the fixed 100-example challenge set.

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print('Best specialist saved to:', FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best specialist saved to: /content/falconsai-t5-bullet-specialist


In [ ]:
# Cell 30 - Inspect unquantized model size

!du -sh /content/falconsai-t5-bullet-specialist
!ls -lh /content/falconsai-t5-bullet-specialist

234M	/content/falconsai-t5-bullet-specialist
total 234M
-rw-r--r-- 1 root root 1.6K Sep 10 08:09 config.json
-rw-r--r-- 1 root root  877 Sep 10 08:09 generation_config.json
-rw------- 1 root root 231M Sep 10 08:09 model.safetensors
-rw-r--r-- 1 root root 2.5K Sep 10 08:09 tokenizer_config.json
-rw-r--r-- 1 root root 2.4M Sep 10 08:09 tokenizer.json
-rw-r--r-- 1 root root 5.3K Sep 10 08:09 training_args.bin


In [ ]:
# Cell 31 - Clear training objects and reload best model in FP16 for inference

# Keep tokenizer and bert_scorer; release Trainer/model training state.
del trainer
del model
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(
    FINAL_MODEL_DIR,
    dtype=torch.float16,
).to('cuda')
model.eval()
model.config.use_cache = True

print('Specialist loaded.')
print('Device:', next(model.parameters()).device)
print('Dtype:', next(model.parameters()).dtype)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Specialist loaded.
Device: cuda:0
Dtype: torch.float16


In [ ]:
# Cell 32 - T5 generation function


def generate_t5_bullets(text, max_new_tokens=256):
    encoder_text = TASK_INSTRUCTION + '\n\nText:\n' + str(text)

    inputs = tokenizer(
        encoder_text,
        return_tensors='pt',
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    inputs = {key: value.to('cuda') for key, value in inputs.items()}

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            no_repeat_ngram_size=3,
        )

    torch.cuda.synchronize()
    latency = time.perf_counter() - start

    raw = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    ).strip()

    prediction = postprocess_generated_text(raw)

    return {
        'output': prediction,
        'latency_seconds': latency,
        'output_tokens': int(outputs.shape[-1]),
    }

In [ ]:
# Cell 33 - Manual sanity test


test_text = """
Vertex Systems reported quarterly revenue of $3.8 billion, up 16% from
the same period last year. Cloud-service revenue increased 28%, while
sales of the company's legacy hardware products declined 7%.

Operating profit rose from $410 million to $475 million, although
operating margin decreased from 18.1% to 17.4% because of higher
infrastructure and energy costs.

The company added 820,000 paying customers during the quarter, bringing
its total customer base to 7.6 million. Customer churn also improved,
falling from 4.8% to 4.1%.

Vertex announced that it will eliminate approximately 600 positions in
its hardware and administrative divisions. The restructuring is expected
to cost about $90 million and should be completed by June next year.
Engineering positions supporting the company's cloud platform will not
be affected.

During the quarter, Vertex completed renovations to its Boston office.
The renovated space includes a larger cafeteria and additional meeting
rooms. Employees also participated in the company's annual charity run,
which raised $180,000 for local organizations.

Vertex introduced a new enterprise security product called Shield Pro.
The product adds automated threat detection and will become generally
available on November 12.

The company raised its full-year revenue guidance from $14.2 billion to
$15.1 billion. Management said demand remains strong in North America
and Asia but warned that enterprise spending weakened in Germany during
August.
"""

result = generate_t5_bullets(test_text)

print(result['output'])
print('\nLatency:', round(result['latency_seconds'], 3), 'seconds')
print('Output tokens:', result['output_tokens'])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


- Vertex Systems reported quarterly revenue of $3.8 billion, up 16% from the same period last year.
- Cloud-service revenue increased 28%, while sales of the company's legacy hardware products declined 7%.
- Operating profit rose from $410 million to $475 million, although operating margin decreased from 18.1% to 17.4% because of higher infrastructure and energy costs.
- The company added 820,000 paying customers during the quarter, bringing its total customer base to 7.6 million.
- Customer churn also improved, falling from 4.8% to $15.1 billion.
- Management said demand remains strong in North America and Asia but warned that enterprise spending weakened in Germany during August.
- Le product adds automated threat detection and will become generally available on November 12.

Latency: 3.327 seconds
Output tokens: 158


In [ ]:
# ============================================================
# HUGGING FACE LOGIN
# ============================================================
#
# In Google Colab:
#
# 1. Open Secrets (key icon)
# 2. Add:
#
#       HF_TOKEN
#
# 3. Paste your Hugging Face WRITE token
#
# 4. Enable notebook access
#
#
# Never hard-code the token into a notebook that may be
# shared.
# ============================================================

from google.colab import userdata

from huggingface_hub import login


HF_TOKEN = ""


login(
    token=HF_TOKEN
)


print(
    "Logged into Hugging Face"
)

Logged into Hugging Face


In [ ]:
# ============================================================
# CREATE HUGGING FACE MODEL REPOSITORY
# ============================================================
#
# exist_ok=True:
#
# rerunning this cell won't fail if the repository already
# exists.
#
#
# private=False:
#
# public model.
#
# Change to:
#
# private=True
#
# if you don't want it publicly visible yet.
# ============================================================

from huggingface_hub import create_repo

HF_REPO_ID = "JayShah07/falconai-text-bullet-t5"

create_repo(

    repo_id=HF_REPO_ID,

    repo_type="model",

    private=False,

    exist_ok=True
)


print(
    "Repository ready:",
    HF_REPO_ID
)

Repository ready: JayShah07/falconai-text-bullet-t5


In [ ]:
# ============================================================
# UPLOAD BEST MODEL
# ============================================================
#
# We are intentionally uploading the clean BEST model,
# NOT all 15 training checkpoints.
#
#
# Hugging Face repo will therefore contain something like:
#
# config.json
# generation_config.json
# model.safetensors
# tokenizer.json
# tokenizer_config.json
# ...
#
#
# Users can later simply do:
#
# AutoModelForCausalLM.from_pretrained(HF_REPO_ID)
#
# ============================================================

from huggingface_hub import upload_folder


upload_folder(

    repo_id=HF_REPO_ID,

    repo_type="model",

    folder_path=FINAL_MODEL_DIR,

    commit_message=(
        "Upload best SmolLM2-135M bullet specialist"
    )
)


print(
    "Uploaded successfully:"
)

print(
    f"https://huggingface.co/{HF_REPO_ID}"
)

Uploaded successfully:
https://huggingface.co/JayShah07/falconai-text-bullet-t5


## Evaluation

In [ ]:
# ============================================================
# FINAL EVALUATION — FALCONSAI T5 BULLET SPECIALIST
# ============================================================
#
# MODEL:
#
#     JayShah07/falconai-text-bullet-t5
#
#
# IMPORTANT:
#
# This notebook evaluates ONLY the T5 model.
#
# DO NOT load:
#
#     SmolLM
#     Qwen
#     another tokenizer
#
# into variables called:
#
#     model
#     tokenizer
#
# before completing the T5 benchmark.
#
#
# WHY?
#
# The previous notebook accidentally did:
#
#     tokenizer = T5 tokenizer
#     model     = T5 model
#
# and later:
#
#     tokenizer = Smol tokenizer
#     model     = Smol model
#
# Therefore generate_t5_bullets() was no longer necessarily
# using the T5 model/tokenizer pair.
#
#
# EVALUATION:
#
#     separate 500-example CSV
#
# Metrics:
#
#     ROUGE-1
#     ROUGE-2
#     ROUGE-L
#
#     BERTScore Precision
#     BERTScore Recall
#     BERTScore F1
#
#     bullet-format compliance
#     predicted bullet count
#     reference bullet count
#     bullet-count error
#
#     prediction compression
#     reference compression
#
#     latency
#     generated tokens
#
# ============================================================

## Evaluation

In [ ]:
# ============================================================
# INSTALL
# ============================================================

!pip install -q -U \
    transformers \
    accelerate \
    sentencepiece \
    pandas \
    tqdm \
    rouge-score \
    bert-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 101.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.


In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import os
import gc
import time
import random

import numpy as np
import pandas as pd

import torch

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

from rouge_score import (
    rouge_scorer
)

from bert_score import (
    BERTScorer
)

In [ ]:
# ============================================================
# GPU CHECK
# ============================================================

print(
    "PyTorch:",
    torch.__version__
)

print(
    "CUDA available:",
    torch.cuda.is_available()
)


if not torch.cuda.is_available():

    raise RuntimeError(
        "Enable a GPU runtime in Google Colab."
    )


print(
    "GPU:",
    torch.cuda.get_device_name(0)
)


print(
    "VRAM:",
    round(
        torch.cuda.get_device_properties(
            0
        ).total_memory
        /
        1024**3,
        2
    ),
    "GB"
)

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
# ============================================================
# EVALUATION CONFIGURATION
# ============================================================

MODEL_ID = (
    "JayShah07/falconai-text-bullet-t5"
)


# CHANGE THIS if your final evaluation CSV has another name.

EVAL_CSV = (
    "/content/output_with_bullet_points.csv"
)


# Must match training.
MAX_INPUT_LENGTH = 2048


# For the current benchmark.
MAX_NEW_TOKENS = 256


SEED = 42


random.seed(
    SEED
)

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)


print(
    "Model:",
    MODEL_ID
)

print(
    "Evaluation CSV:",
    EVAL_CSV
)

Model: JayShah07/falconai-text-bullet-t5
Evaluation CSV: /content/output_with_bullet_points.csv


In [ ]:
# ============================================================
# EXACT TASK INSTRUCTION USED FOR TRAINING
# ============================================================

TASK_INSTRUCTION = """
Convert the following English text into concise bullet points containing all materially important information.

Follow these rules:

- Extract all important and independently useful points.
- The number of bullets must depend entirely on the information in the text.
- Never use a fixed number of bullets.
- Use one bullet for each distinct important point.
- Combine details that naturally belong together.
- Remove repetition, filler, metadata, boilerplate, and trivial details.
- Do not repeat the same information in multiple bullets.
- Preserve important names, dates, numbers, quantities, comparisons, causes, conditions, decisions, and conclusions.
- Do not add, infer, or assume information that is not supported by the source text.
- Do not turn contextual information into new advice or recommendations.
- Keep every bullet concise while preserving the original meaning.
- Return only bullet points.
- Start every bullet with "- ".

Text:
""".strip()


print(
    TASK_INSTRUCTION
)

Convert the following English text into concise bullet points containing all materially important information.

Follow these rules:

- Extract all important and independently useful points.
- The number of bullets must depend entirely on the information in the text.
- Never use a fixed number of bullets.
- Use one bullet for each distinct important point.
- Combine details that naturally belong together.
- Remove repetition, filler, metadata, boilerplate, and trivial details.
- Do not repeat the same information in multiple bullets.
- Preserve important names, dates, numbers, quantities, comparisons, causes, conditions, decisions, and conclusions.
- Do not add, infer, or assume information that is not supported by the source text.
- Do not turn contextual information into new advice or recommendations.
- Keep every bullet concise while preserving the original meaning.
- Return only bullet points.
- Start every bullet with "- ".

Text:


In [ ]:
# ============================================================
# LOAD T5 TOKENIZER
# ============================================================

tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_ID
    )
)


print(
    "Tokenizer loaded:",
    tokenizer.__class__.__name__
)


print(
    "Vocabulary size:",
    len(tokenizer)
)


print(
    "PAD token:",
    tokenizer.pad_token
)


print(
    "EOS token:",
    tokenizer.eos_token
)

Tokenizer loaded: T5Tokenizer
Vocabulary size: 32101
PAD token: <pad>
EOS token: </s>


In [ ]:
# ============================================================
# VERIFY BULLET SPECIAL TOKEN
# ============================================================

BULLET_TOKEN = (
    "<BULLET>"
)


bullet_token_id = (
    tokenizer.convert_tokens_to_ids(
        BULLET_TOKEN
    )
)


print(
    "BULLET token:",
    BULLET_TOKEN
)


print(
    "BULLET token ID:",
    bullet_token_id
)


print(
    "UNK token ID:",
    tokenizer.unk_token_id
)


assert (
    bullet_token_id
    !=
    tokenizer.unk_token_id
), (
    "<BULLET> is not present in the downloaded tokenizer. "
    "Stop evaluation and fix the uploaded tokenizer/model."
)


print(
    "<BULLET> token is correctly available."
)

BULLET token: <BULLET>
BULLET token ID: 32100
UNK token ID: 2
<BULLET> token is correctly available.


In [ ]:
# ============================================================
# LOAD T5 SPECIALIST
# ============================================================

model = (
    AutoModelForSeq2SeqLM
    .from_pretrained(

        MODEL_ID,

        dtype=torch.float16
    )
)


model = model.to(
    "cuda"
)


model.eval()


print(
    "Loaded:",
    MODEL_ID
)


print(
    "Architecture:",
    model.__class__.__name__
)


print(
    "Encoder-decoder:",
    model.config.is_encoder_decoder
)


print(
    "Device:",
    next(
        model.parameters()
    ).device
)


print(
    "Dtype:",
    next(
        model.parameters()
    ).dtype
)


assert (
    model.config.is_encoder_decoder
), (
    "Wrong model loaded. Expected an encoder-decoder T5 model."
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loaded: JayShah07/falconai-text-bullet-t5
Architecture: T5ForConditionalGeneration
Encoder-decoder: True
Device: cuda:0
Dtype: torch.float16


In [ ]:
# ============================================================
# MODEL / TOKENIZER SANITY CHECK
# ============================================================

print(
    "Tokenizer vocabulary:",
    len(tokenizer)
)


print(
    "Model embedding vocabulary:",
    model.get_input_embeddings().weight.shape[0]
)


assert (
    len(tokenizer)
    <=
    model.get_input_embeddings().weight.shape[0]
), (
    "Tokenizer contains more tokens than model embeddings."
)


print(
    "Model/tokenizer compatibility looks correct."
)

Tokenizer vocabulary: 32101
Model embedding vocabulary: 32101
Model/tokenizer compatibility looks correct.


In [ ]:
# ============================================================
# BULLET REPRESENTATION HELPERS
# ============================================================


def reference_to_internal(
    text
):

    """
    Convert:

        - point one
        - point two

    into:

        <BULLET> point one <BULLET> point two

    Used only when we need the internal representation.
    """

    lines = [

        line.strip()

        for line in str(text).splitlines()

        if line.strip()
    ]


    bullets = []


    for line in lines:

        if line.startswith(
            "- "
        ):

            line = line[2:].strip()


        if line:

            bullets.append(
                line
            )


    return (
        " "
        .join(

            BULLET_TOKEN
            +
            " "
            +
            bullet

            for bullet in bullets
        )
    )



def postprocess_generated_text(
    raw_text
):

    """
    Convert T5 internal output:

        <BULLET> point one <BULLET> point two

    into:

        - point one
        - point two
    """

    text = str(
        raw_text
    ).strip()


    # --------------------------------------------------------
    # If model generated our internal marker, split on it.
    # --------------------------------------------------------

    if BULLET_TOKEN in text:

        pieces = [

            piece.strip()

            for piece in text.split(
                BULLET_TOKEN
            )

            if piece.strip()
        ]


        return "\n".join(

            "- " + piece

            for piece in pieces
        )


    # --------------------------------------------------------
    # If the model already generated proper newline bullets,
    # preserve them.
    # --------------------------------------------------------

    lines = [

        line.strip()

        for line in text.splitlines()

        if line.strip()
    ]


    if lines and all(

        line.startswith("- ")

        for line in lines
    ):

        return "\n".join(
            lines
        )


    # --------------------------------------------------------
    # Otherwise treat the generated text as one bullet.
    # --------------------------------------------------------

    if text:

        return (
            "- "
            +
            text
        )


    return ""

In [ ]:
# ============================================================
# BUILD EXACT T5 ENCODER INPUT
# ============================================================


def build_encoder_text(
    text
):

    return (

        TASK_INSTRUCTION

        +

        "\n"

        +

        str(text).strip()
    )

In [ ]:
# ============================================================
# CORRECT T5 GENERATION FUNCTION
# ============================================================


def generate_t5_bullets(

    text,

    max_new_tokens=MAX_NEW_TOKENS
):

    # --------------------------------------------------------
    # Build ENCODER input only.
    # --------------------------------------------------------

    encoder_text = (
        build_encoder_text(
            text
        )
    )


    inputs = tokenizer(

        encoder_text,

        return_tensors="pt",

        max_length=MAX_INPUT_LENGTH,

        truncation=True
    )


    inputs = {

        key:
            value.to(
                "cuda"
            )

        for key, value
        in inputs.items()
    }


    input_tokens = int(
        inputs[
            "input_ids"
        ].shape[-1]
    )


    torch.cuda.synchronize()


    start = (
        time.perf_counter()
    )


    with torch.inference_mode():

        generated_ids = (
            model.generate(

                **inputs,

                max_new_tokens=(
                    max_new_tokens
                ),

                do_sample=False,

                num_beams=1,

                no_repeat_ngram_size=3
            )
        )


    torch.cuda.synchronize()


    latency = (

        time.perf_counter()

        -

        start
    )


    # ========================================================
    # IMPORTANT
    # ========================================================
    #
    # T5 is encoder-decoder.
    #
    # generated_ids contains DECODER output.
    #
    # DO NOT slice it using input length.
    #
    # That slicing pattern belongs to decoder-only models.
    #
    # ========================================================


    raw_output = tokenizer.decode(

        generated_ids[0],

        skip_special_tokens=False
    )


    # --------------------------------------------------------
    # Remove T5 PAD/EOS while retaining <BULLET>.
    # --------------------------------------------------------

    if tokenizer.pad_token:

        raw_output = (
            raw_output.replace(
                tokenizer.pad_token,
                ""
            )
        )


    if tokenizer.eos_token:

        raw_output = (
            raw_output.replace(
                tokenizer.eos_token,
                ""
            )
        )


    raw_output = (
        raw_output.strip()
    )


    prediction = (
        postprocess_generated_text(
            raw_output
        )
    )


    # --------------------------------------------------------
    # Count decoder-generated non-PAD tokens.
    # --------------------------------------------------------

    generated_token_count = int(

        (
            generated_ids[0]

            !=

            tokenizer.pad_token_id
        )

        .sum()

        .item()
    )


    return {

        "output":
            prediction,

        "raw_output":
            raw_output,

        "latency_seconds":
            latency,

        "output_tokens":
            generated_token_count,

        "input_tokens":
            input_tokens
    }

In [ ]:
# ============================================================
# MANDATORY SANITY TEST
# ============================================================

test_text = """
Acme reported quarterly revenue of $4.2 billion, up 12% year over year.

Operating profit increased 8% to $620 million, although operating margin
declined from 17.2% to 14.8%.

The company added 1.3 million customers during the quarter and raised
full-year revenue guidance from $16 billion to $17.5 billion.

Management warned that European demand weakened in July.
"""


result = generate_t5_bullets(
    test_text
)


print(
    "RAW DECODER OUTPUT:"
)


print(
    repr(
        result[
            "raw_output"
        ]
    )
)


print()


print(
    "FINAL OUTPUT:"
)


print(
    result[
        "output"
    ]
)


print()


print(
    "Input tokens:",
    result[
        "input_tokens"
    ]
)


print(
    "Output tokens:",
    result[
        "output_tokens"
    ]
)


print(
    "Latency:",
    round(
        result[
            "latency_seconds"
        ],
        3
    ),
    "seconds"
)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAW DECODER OUTPUT:
'<BULLET> Acme reported quarterly revenue of $4.2 billion, up 12% year over year.<BULLET> Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.<BULLET> The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.<BULLET> Management warned that European demand weakened in July.'

FINAL OUTPUT:
- Acme reported quarterly revenue of $4.2 billion, up 12% year over year.
- Operating profit increased 8% to $620 million, although operating margin declined from 17.2% to 14.8%.
- The company added 1.3 million customers during the quarter and raised full-year revenue guidance from $16 billion to $17.5 billion.
- Management warned that European demand weakened in July.

Input tokens: 279
Output tokens: 79
Latency: 3.891 seconds


In [ ]:
# ============================================================
# REQUIRED EVALUATION COLUMNS
# ============================================================

REQUIRED_COLUMNS = [

    "text",

    "source",

    "example_id",

    "bullet_points"
]

In [ ]:
# ============================================================
# LOAD FINAL EVALUATION CSV
# ============================================================


if not os.path.exists(
    EVAL_CSV
):

    raise FileNotFoundError(

        f"Evaluation CSV not found: {EVAL_CSV}"
    )


final_eval_df = pd.read_csv(
    EVAL_CSV
)


print(
    "Rows:",
    len(final_eval_df)
)


print(
    "Columns:",
    final_eval_df.columns.tolist()
)


display(
    final_eval_df.head()
)

Rows: 500
Columns: ['text', 'source', 'example_id', 'bullet_points']


,text,source,example_id,bullet_points
0,Powell: N. Korea Blast Not Nuclear Event The U...,ag_news,0,- Powell says the large North Korean explosion...
1,Jerusalem (CNN) -- Two attacks carried out aga...,cnn_dailymail,1,- Two recent attacks on Palestinians sparked I...
2,Former Kan. Junior College Coach Indicted (AP)...,ag_news,2,- Former Kansas junior college basketball coac...
3,The Ethiopian Airlines flight was travelling f...,xsum,3,- Ethiopian Airlines Flight ET500 was travelin...
4,"Minami Sanriku, Japan (CNN) -- A 60-year-old ...",cnn_dailymail,4,"- A 60‑year‑old man, Hiromitsu Shinkawa, was r..."


In [ ]:
# ============================================================
# CLEAN EVALUATION DATA
# ============================================================


def clean_dataframe(
    dataframe
):

    dataframe = (
        dataframe.copy()
    )


    dataframe = (
        dataframe.dropna(

            subset=[
                "text",
                "bullet_points"
            ]
        )
    )


    dataframe[
        "text"
    ] = (

        dataframe[
            "text"
        ]

        .astype(str)

        .str.strip()
    )


    dataframe[
        "bullet_points"
    ] = (

        dataframe[
            "bullet_points"
        ]

        .astype(str)

        .str.strip()
    )


    dataframe = dataframe[

        (
            dataframe[
                "text"
            ]
            !=
            ""
        )

        &

        (
            dataframe[
                "bullet_points"
            ]
            !=
            ""
        )
    ]


    return (

        dataframe

        .reset_index(
            drop=True
        )
    )


final_eval_df = clean_dataframe(
    final_eval_df
)


for column in REQUIRED_COLUMNS:

    assert (
        column
        in
        final_eval_df.columns
    ), (
        f"Missing column: {column}"
    )


print(
    "Clean evaluation rows:",
    len(final_eval_df)
)

Clean evaluation rows: 500


In [ ]:
# ============================================================
# REFERENCE BULLET DISTRIBUTION
# ============================================================


def count_bullets(
    text
):

    return sum(

        line.strip().startswith(
            "- "
        )

        for line in str(
            text
        ).splitlines()
    )


final_eval_df[
    "reference_bullet_count"
] = (

    final_eval_df[
        "bullet_points"
    ]

    .apply(
        count_bullets
    )
)


print(
    final_eval_df[
        "reference_bullet_count"
    ]
    .value_counts()
    .sort_index()
)


print()


print(
    "Average reference bullets:",
    final_eval_df[
        "reference_bullet_count"
    ].mean()
)

reference_bullet_count
1     64
2     82
3     34
4     31
5     71
6     92
8     84
10    40
18     1
21     1
Name: count, dtype: int64

Average reference bullets: 4.944


In [ ]:
# ============================================================
# TEST ONE ACTUAL EVALUATION EXAMPLE
# ============================================================

row = final_eval_df.iloc[
    0
]


result = generate_t5_bullets(
    row[
        "text"
    ]
)


print(
    "=" * 100
)


print(
    "SOURCE:",
    row[
        "source"
    ]
)


print(
    "\nINPUT:\n"
)


print(
    row[
        "text"
    ]
)


print(
    "\nREFERENCE:\n"
)


print(
    row[
        "bullet_points"
    ]
)


print(
    "\nRAW T5 OUTPUT:\n"
)


print(
    repr(
        result[
            "raw_output"
        ]
    )
)


print(
    "\nFINAL T5 OUTPUT:\n"
)


print(
    result[
        "output"
    ]
)


print()


print(
    "Predicted bullets:",
    count_bullets(
        result[
            "output"
        ]
    )
)


print(
    "Reference bullets:",
    count_bullets(
        row[
            "bullet_points"
        ]
    )
)


print(
    "Input tokens:",
    result[
        "input_tokens"
    ]
)


print(
    "Output tokens:",
    result[
        "output_tokens"
    ]
)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SOURCE: ag_news

INPUT:

Powell: N. Korea Blast Not Nuclear Event The United States does not believe that a large explosion in North Korea was related to the communist country #39;s suspected nuclear weapons program, President Bush #39;s foreign policy advisers said Sunday.

REFERENCE:

- Powell says the large North Korean explosion was not a nuclear event.  
- U.S. officials do not believe the blast was linked to North Korea’s suspected nuclear weapons program.  
- The statement was made by President Bush’s foreign policy advisers on Sunday.

RAW T5 OUTPUT:

'<BULLET> The United States does not believe that a large explosion in North Korea was related to the communist country #39;s suspected nuclear weapons program, President Bush #39);s foreign policy advisers said Sunday.'

FINAL T5 OUTPUT:

- The United States does not believe that a large explosion in North Korea was related to the communist country #39;s suspected nuclear weapons program, President Bush #39);s foreign policy advi

In [ ]:
# ============================================================
# GENERATE T5 OUTPUTS FOR FINAL EVALUATION
# ============================================================

predictions = []


for _, row in tqdm(

    final_eval_df.iterrows(),

    total=len(
        final_eval_df
    ),

    desc="Falconsai-T5-Bullet-SFT"
):

    try:

        result = (
            generate_t5_bullets(

                row[
                    "text"
                ]
            )
        )


        predictions.append({

            "example_id":
                row[
                    "example_id"
                ],

            "source":
                row[
                    "source"
                ],

            "text":
                row[
                    "text"
                ],

            "reference":
                row[
                    "bullet_points"
                ],

            "prediction":
                result[
                    "output"
                ],

            "raw_output":
                result[
                    "raw_output"
                ],

            "input_tokens":
                result[
                    "input_tokens"
                ],

            "output_tokens":
                result[
                    "output_tokens"
                ],

            "latency_seconds":
                result[
                    "latency_seconds"
                ]
        })


    except Exception as error:

        predictions.append({

            "example_id":
                row[
                    "example_id"
                ],

            "source":
                row[
                    "source"
                ],

            "text":
                row[
                    "text"
                ],

            "reference":
                row[
                    "bullet_points"
                ],

            "prediction":
                "",

            "raw_output":
                "",

            "input_tokens":
                None,

            "output_tokens":
                None,

            "latency_seconds":
                None,

            "error":
                str(
                    error
                )
        })


t5_results = pd.DataFrame(
    predictions
)


print(
    "Generated:",
    len(t5_results)
)


display(
    t5_results.head()
)

Falconsai-T5-Bullet-SFT:   0%|          | 0/500 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Generated: 500


,example_id,source,text,reference,prediction,raw_output,input_tokens,output_tokens,latency_seconds
0,0,ag_news,Powell: N. Korea Blast Not Nuclear Event The U...,- Powell says the large North Korean explosion...,- The United States does not believe that a la...,<BULLET> The United States does not believe th...,258,45,1.270487
1,1,cnn_dailymail,Jerusalem (CNN) -- Two attacks carried out aga...,- Two recent attacks on Palestinians sparked I...,- The latest assaults are not the only inciden...,<BULLET> The latest assaults are not the only ...,2048,147,4.377333
2,2,ag_news,Former Kan. Junior College Coach Indicted (AP)...,- Former Kansas junior college basketball coac...,- Former Kan.\n- Junior College Coach Indicted...,<BULLET> Former Kan.<BULLET> Junior College Co...,253,15,0.512985
3,3,xsum,The Ethiopian Airlines flight was travelling f...,- Ethiopian Airlines Flight ET500 was travelin...,- The Ethiopian Airlines flight was travelling...,<BULLET> The Ethiopian Airlines flight was tra...,264,63,1.297209
4,4,cnn_dailymail,"Minami Sanriku, Japan (CNN) -- A 60-year-old ...","- A 60‑year‑old man, Hiromitsu Shinkawa, was r...","- Prime Minister Naoto Kan said about 15,000 p...",<BULLET> Prime Minister Naoto Kan said about 1...,1313,137,7.795559


In [ ]:
# ============================================================
# SAVE RAW PREDICTIONS BEFORE SCORING
# ============================================================

RAW_RESULTS_FILE = (
    "/content/falconai_t5_final_raw_predictions.csv"
)


t5_results.to_csv(

    RAW_RESULTS_FILE,

    index=False
)


print(
    "Saved:",
    RAW_RESULTS_FILE
)

Saved: /content/falconai_t5_final_raw_predictions.csv


In [ ]:
# ============================================================
# PRE-METRIC SANITY CHECK
# ============================================================


def word_count(
    text
):

    return len(
        str(text).split()
    )



t5_results[
    "predicted_bullets"
] = (

    t5_results[
        "prediction"
    ]

    .apply(
        count_bullets
    )
)


t5_results[
    "reference_bullets"
] = (

    t5_results[
        "reference"
    ]

    .apply(
        count_bullets
    )
)


t5_results[
    "input_words"
] = (

    t5_results[
        "text"
    ]

    .apply(
        word_count
    )
)


t5_results[
    "prediction_words"
] = (

    t5_results[
        "prediction"
    ]

    .apply(
        word_count
    )
)


t5_results[
    "reference_words"
] = (

    t5_results[
        "reference"
    ]

    .apply(
        word_count
    )
)


print(
    "Average predicted bullets:",
    t5_results[
        "predicted_bullets"
    ].mean()
)


print(
    "Average reference bullets:",
    t5_results[
        "reference_bullets"
    ].mean()
)


print(
    "Average input words:",
    t5_results[
        "input_words"
    ].mean()
)


print(
    "Average prediction words:",
    t5_results[
        "prediction_words"
    ].mean()
)


print(
    "Average reference words:",
    t5_results[
        "reference_words"
    ].mean()
)


print(
    "Average generated tokens:",
    t5_results[
        "output_tokens"
    ].mean()
)

Average predicted bullets: 3.586
Average reference bullets: 4.944
Average input words: 293.358
Average prediction words: 64.492
Average reference words: 113.818
Average generated tokens: 90.008


In [ ]:
# ============================================================
# HUMAN SANITY CHECK
# ============================================================

sample = t5_results.sample(

    n=min(
        5,
        len(t5_results)
    ),

    random_state=42
)


for _, row in sample.iterrows():

    print(
        "=" * 100
    )


    print(
        "\nINPUT:\n"
    )


    print(
        row[
            "text"
        ]
    )


    print(
        "\nREFERENCE:\n"
    )


    print(
        row[
            "reference"
        ]
    )


    print(
        "\nT5:\n"
    )


    print(
        row[
            "prediction"
        ]
    )


    print()


    print(
        "Bullets:",
        row[
            "predicted_bullets"
        ],
        "/",
        row[
            "reference_bullets"
        ]
    )


    print(
        "Output tokens:",
        row[
            "output_tokens"
        ]
    )


    print()


INPUT:

Fighting to Save Plan Mayor Anthony Williams, pushing for baseball on the Anacostia, has canceled a meeting Monday with the D.C. Council he must sway.

REFERENCE:

- Fighting to Save Plan Mayor Anthony Williams, pushing for baseball on the Anacostia, has canceled a meeting Monday with the D.C. Council he must sway.

T5:

- Fighting to Save Plan Mayor Anthony Williams, pushing for baseball on the Anacostia, has canceled a meeting Monday with the D.C. Council he must sway.

Bullets: 1 / 1
Output tokens: 41


INPUT:

Brockton tops Eagles BROCKTON -- A new era? Nah. Just the same old Brockton coached by a different Colombo.

REFERENCE:

- Just the same old Brockton coached by a different Colombo.

T5:

- Brockton tops Eagles BROCKTON -- A new era? Nah.

Bullets: 1 / 1
Output tokens: 21


INPUT:

FCC Takes VoIP Regulation Out of State's Hands Decision exempts Vonage from Minnesota state telephony laws.

REFERENCE:

- FCC Takes VoIP Regulation Out of State's Hands Decision exempts V

In [ ]:
# ============================================================
# EVALUATORS
# ============================================================

rouge = (
    rouge_scorer.RougeScorer(

        [
            "rouge1",
            "rouge2",
            "rougeL"
        ],

        use_stemmer=True
    )
)


bert_scorer = (
    BERTScorer(

        model_type=(
            "distilbert-base-uncased"
        ),

        lang="en",

        device="cuda",

        rescale_with_baseline=False
    )
)


print(
    "Evaluators ready."
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluators ready.


In [ ]:
# ============================================================
# BULLET FORMAT
# ============================================================


def bullet_format_score(
    text
):

    lines = [

        line.strip()

        for line in str(
            text
        ).splitlines()

        if line.strip()
    ]


    if not lines:

        return 0.0


    correct = sum(

        line.startswith(
            "- "
        )

        for line in lines
    )


    return (

        correct

        /

        len(lines)
    )

In [ ]:
# ============================================================
# FULL FINAL SCORING
# ============================================================


def score_t5_results(
    results_df
):

    scored = (
        results_df.copy()
    )


    # ========================================================
    # ROUGE
    # ========================================================

    rouge1_values = []

    rouge2_values = []

    rougeL_values = []


    for _, row in tqdm(

        scored.iterrows(),

        total=len(scored),

        desc="ROUGE"
    ):

        scores = rouge.score(

            row[
                "reference"
            ],

            row[
                "prediction"
            ]
        )


        rouge1_values.append(
            scores[
                "rouge1"
            ].fmeasure
        )


        rouge2_values.append(
            scores[
                "rouge2"
            ].fmeasure
        )


        rougeL_values.append(
            scores[
                "rougeL"
            ].fmeasure
        )


    scored[
        "rouge1"
    ] = rouge1_values


    scored[
        "rouge2"
    ] = rouge2_values


    scored[
        "rougeL"
    ] = rougeL_values


    # ========================================================
    # BERTSCORE
    # ========================================================

    P, R, F1 = (
        bert_scorer.score(

            scored[
                "prediction"
            ].tolist(),

            scored[
                "reference"
            ].tolist()
        )
    )


    scored[
        "bertscore_precision"
    ] = (
        P.cpu().numpy()
    )


    scored[
        "bertscore_recall"
    ] = (
        R.cpu().numpy()
    )


    scored[
        "bertscore_f1"
    ] = (
        F1.cpu().numpy()
    )


    # ========================================================
    # FORMAT
    # ========================================================

    scored[
        "bullet_format"
    ] = (

        scored[
            "prediction"
        ]

        .apply(
            bullet_format_score
        )
    )


    # ========================================================
    # BULLET COUNTS
    # ========================================================

    scored[
        "predicted_bullets"
    ] = (

        scored[
            "prediction"
        ]

        .apply(
            count_bullets
        )
    )


    scored[
        "reference_bullets"
    ] = (

        scored[
            "reference"
        ]

        .apply(
            count_bullets
        )
    )


    scored[
        "bullet_count_error"
    ] = (

        scored[
            "predicted_bullets"
        ]

        -

        scored[
            "reference_bullets"
        ]

    ).abs()


    # ========================================================
    # WORD COUNTS
    # ========================================================

    scored[
        "input_words"
    ] = (

        scored[
            "text"
        ]

        .apply(
            word_count
        )
    )


    scored[
        "prediction_words"
    ] = (

        scored[
            "prediction"
        ]

        .apply(
            word_count
        )
    )


    scored[
        "reference_words"
    ] = (

        scored[
            "reference"
        ]

        .apply(
            word_count
        )
    )


    safe_input = (

        scored[
            "input_words"
        ]

        .clip(
            lower=1
        )
    )


    # ========================================================
    # COMPRESSION
    # ========================================================

    scored[
        "compression_ratio"
    ] = (

        scored[
            "prediction_words"
        ]

        /

        safe_input
    )


    scored[
        "reference_compression_ratio"
    ] = (

        scored[
            "reference_words"
        ]

        /

        safe_input
    )


    # ========================================================
    # SUMMARY
    # ========================================================

    summary = {

        "model":
            "Falconsai-T5-Bullet-SFT",

        "examples":
            len(scored),

        "rouge1":
            scored[
                "rouge1"
            ].mean(),

        "rouge2":
            scored[
                "rouge2"
            ].mean(),

        "rougeL":
            scored[
                "rougeL"
            ].mean(),

        "bertscore_precision":
            scored[
                "bertscore_precision"
            ].mean(),

        "bertscore_recall":
            scored[
                "bertscore_recall"
            ].mean(),

        "bertscore_f1":
            scored[
                "bertscore_f1"
            ].mean(),

        "bullet_format":
            scored[
                "bullet_format"
            ].mean(),

        "avg_predicted_bullets":
            scored[
                "predicted_bullets"
            ].mean(),

        "avg_reference_bullets":
            scored[
                "reference_bullets"
            ].mean(),

        "mean_bullet_count_error":
            scored[
                "bullet_count_error"
            ].mean(),

        "compression_ratio":
            scored[
                "compression_ratio"
            ].mean(),

        "reference_compression_ratio":
            scored[
                "reference_compression_ratio"
            ].mean(),

        "avg_latency_seconds":
            scored[
                "latency_seconds"
            ].mean(),

        "avg_output_tokens":
            scored[
                "output_tokens"
            ].mean()
    }


    return (
        scored,
        summary
    )

In [ ]:
# ============================================================
# SCORE MODEL
# ============================================================

t5_scored, t5_summary = (
    score_t5_results(
        t5_results
    )
)


summary_df = pd.DataFrame(
    [t5_summary]
)


display(
    summary_df.T
)

ROUGE:   0%|          | 0/500 [00:00<?, ?it/s]

,0
model,Falconsai-T5-Bullet-SFT
examples,500
rouge1,0.604042
rouge2,0.532914
rougeL,0.560059
bertscore_precision,0.912761
bertscore_recall,0.85685
bertscore_f1,0.882576
bullet_format,1.0
avg_predicted_bullets,3.586


In [ ]:
# ============================================================
# SAVE RESULTS
# ============================================================

SCORED_FILE = (
    "/content/falconai_t5_final_scored.csv"
)


SUMMARY_FILE = (
    "/content/falconai_t5_final_summary.csv"
)


t5_scored.to_csv(

    SCORED_FILE,

    index=False
)


summary_df.to_csv(

    SUMMARY_FILE,

    index=False
)


print(
    "Scored predictions:",
    SCORED_FILE
)


print(
    "Summary:",
    SUMMARY_FILE
)

Scored predictions: /content/falconai_t5_final_scored.csv
Summary: /content/falconai_t5_final_summary.csv


In [ ]:
# ============================================================
# FAILURE ANALYSIS
# ============================================================

worst = (

    t5_scored

    .sort_values(
        "rougeL",
        ascending=True
    )

    .head(10)
)


for _, row in worst.iterrows():

    print(
        "=" * 100
    )


    print(
        "\nSOURCE:",
        row[
            "source"
        ]
    )


    print(
        "\nINPUT:\n"
    )


    print(
        row[
            "text"
        ]
    )


    print(
        "\nREFERENCE:\n"
    )


    print(
        row[
            "reference"
        ]
    )


    print(
        "\nT5:\n"
    )


    print(
        row[
            "prediction"
        ]
    )


    print()


    print(
        "ROUGE-L:",
        round(
            row[
                "rougeL"
            ],
            4
        )
    )


    print(
        "BERTScore F1:",
        round(
            row[
                "bertscore_f1"
            ],
            4
        )
    )


    print(
        "Bullet count:",
        row[
            "predicted_bullets"
        ],
        "/",
        row[
            "reference_bullets"
        ]
    )


    print()


SOURCE: email

INPUT:

Hi Kent,  I was wondering if we could schedule a conference call to discuss the form of  these agreements.
Are you available any time soon?
Thanks,

REFERENCE:

- Hi Kent, I was wondering if we could schedule a conference call to discuss the form of these agreements.

T5:

- Are you available any time soon?

ROUGE-L: 0.0
BERTScore F1: 0.6912
Bullet count: 1 / 1


SOURCE: email

INPUT:

Any news on looking at the NiSource storage stuff?
Have we gotten the packet  from Chase or whichever investment banker guys you talked to?
I heard again  from a non-Enron source that "Enron is looking at the MHP assets" so it must  be true!
Thanks.
DF

REFERENCE:

- Any news on looking at the NiSource storage stuff?
- Have we gotten the packet from Chase or whichever investment banker guys you talked to?
- I heard again from a non-Enron source that "Enron is looking at the MHP assets" so it must be true!

T5:

- DF

ROUGE-L: 0.0
BERTScore F1: 0.6436
Bullet count: 1 / 3


SOURCE: 